# UC2 · 03 — What the representation buys, and whether it holds

Four checks. One measures the cost of *not* having hyperedges. Two assert the
invariants `01` promised by construction. One compares the responsive graph
against an independent mechanistic network solved by the study's own CARNIVAL
pipeline.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

import annnet as an

DATA = Path("data")
SEED = 7
TIMES = ("1h", "12h", "24h", "48h", "72h", "96h")
EARLY, LATE = ("1h", "12h", "24h"), ("48h", "72h", "96h")

G = an.AnnNet.read(str(DATA / "uc2.annnet"))

edge_kind = G.attrs.get_attr_from_edges("edge_kind")
E = pd.DataFrame(
    [(e, s[0], *s[1], t[0], *t[1], edge_kind[e]) for e, (s, t, _) in G.edge_definitions.items()],
    columns=["edge_id", "src", "src_mech", "src_time", "tgt", "tgt_mech", "tgt_time", "edge_kind"],
)
intra = E[(E.src_mech == E.tgt_mech) & (E.src_time == E.tgt_time)]
gated = intra[intra.edge_kind.isin(["signaling", "regulatory"])].assign(
    src_symbol=lambda d: d.src.str.split(":").str[1],
    tgt_symbol=lambda d: d.tgt.str.split(":").str[1],
)

## Native hyperedges vs. clique expansion

A tool without a hyperedge primitive expands each one into n(n−1)/2 pairwise
edges, and drops the signed metabolic stoichiometry, which has no pairwise
encoding at all. The counts below are what that costs here.

The graph still materialises as a plain NetworkX object whenever a caller wants
one. `to_nx` returns it alongside a manifest that carries the parts NetworkX has
no slot for — hyperedges with their per-endpoint coefficients, and the slices —
so the expansion is a choice rather than a property of the format.

In [2]:
nx_graph, manifest = an.to_nx(G)
print(f"as networkx: {type(nx_graph).__name__} with {nx_graph.number_of_nodes():,} nodes"
      f" and {nx_graph.number_of_edges():,} edges")
print(f"manifest keys: {sorted(manifest)}")

sizes = [
    len(set(s.get("head", [])) | set(s.get("tail", [])) | set(s.get("members", [])))
    for s in G.hyperedge_definitions.values()
]
clique = sum(n * (n - 1) // 2 for n in sizes if n >= 2)
native = G.global_count("edges")
expanded = native - len(sizes) + clique
print(f"AnnNet native edges   : {native:,}  ({len(sizes):,} hyperedges)")
print(f"clique-expanded edges : {expanded:,}  ({expanded / native:.2f}x, signed stoichiometry lost)")

as networkx: MultiDiGraph with 33,733 nodes and 47,474 edges
manifest keys: ['edge_attrs', 'edge_directed', 'edges', 'manifest_version', 'multilayer', 'slice_weights', 'slices', 'vertex_attrs', 'weights']
AnnNet native edges   : 63,891  (16,417 hyperedges)
clique-expanded edges : 319,826  (5.01x, signed stoichiometry lost)


## Construction invariants

The **responsive invariant** checks the 0-hop gate: every intra-layer signaling
or regulatory edge has both endpoints responsive at its own timepoint. The
**time invariant** checks that every time-coupling edge links one entity to
itself across two consecutive layers.

In [3]:
responsive_at = {
    t: set(g["symbol"])
    for t, g in pl.read_parquet(DATA / "responsive.parquet").to_pandas().groupby("time", observed=True)
}
violations = sum(
    r.src.split(":", 1)[1] not in responsive_at[r.src_time]
    or r.tgt.split(":", 1)[1] not in responsive_at[r.tgt_time]
    for r in gated.itertuples()
)
print(f"responsive invariant : {violations} violations out of {len(gated):,} gated edges")

order = {t: i for i, t in enumerate(TIMES)}
coupling = E[E.edge_kind == "coupling_time"]
consecutive = sum(
    r.src == r.tgt and order[r.tgt_time] - order[r.src_time] == 1 for r in coupling.itertuples()
)
print(f"time invariant       : {consecutive:,}/{len(coupling):,} same-entity and consecutive")

responsive invariant : 0 violations out of 36,492 gated edges
time invariant       : 7,660/7,660 same-entity and consecutive


## CARNIVAL recovery

How much of the study's independently-solved early/late network the responsive
graph already contains. The graph is raw prior knowledge plus the response
gate, with no optimisation of its own, so overlap is evidence the gate selects
a relevant subnetwork. It is a sanity check rather than a benchmark: UC2 uses
OmniPath + DoRothEA, while CARNIVAL solved over a richer merged prior.

In [4]:
carnival = pl.read_parquet(DATA / "carnival_network.parquet").to_pandas()

for network, window in [("early", EARLY), ("late", LATE)]:
    rows = gated[gated.src_time.isin(window)]
    present = {frozenset((a, b)) for a, b in zip(rows.src_symbol, rows.tgt_symbol)}
    solved = {frozenset((r.source, r.target)) for r in carnival[carnival.network == network].itertuples()}
    hit = len(solved & present)
    print(f"{network:>5}: {hit:>3}/{len(solved)} solved edges present ({hit / len(solved):.0%})"
          f" | graph pairs in window: {len(present):,}")

early:   9/149 solved edges present (6%) | graph pairs in window: 4,579
 late:  45/190 solved edges present (24%) | graph pairs in window: 15,149


## Null-DoRothEA permutation

The regulatory layer at the last timepoint is rewired at random and Q1 re-run,
50 times. A random layer of the same size should not recover the real
(TF, complex) pairs.

In [5]:
MIN_FRACTION, MIN_HITS, N_PERMUTATIONS = 0.5, 2, 50

subunits = {
    e: {m[0].removeprefix("prot:") for m in spec["members"]}
    for e, spec in G.hyperedge_definitions.items()
    if str(e).startswith("cpx:")
}
subunits = {e: g for e, g in subunits.items() if len(g) >= 3}
late = gated[(gated.edge_kind == "regulatory") & (gated.src_time == TIMES[-1])]


def coregulated(targets_of):
    """(TF, complex) pairs passing the Q1 coverage rule."""
    return {
        (tf, complex_id)
        for complex_id, genes in subunits.items()
        for tf, targets in targets_of.items()
        if len(genes & targets) >= MIN_HITS and len(genes & targets) / len(genes) >= MIN_FRACTION
    }


real_targets = late.groupby("src_symbol")["tgt_symbol"].apply(set)
real = coregulated(real_targets)

tfs = sorted(real_targets.index)
universe = sorted({g for s in subunits.values() for g in s}.union(*real_targets.values))
overlaps = []
for k in range(N_PERMUTATIONS):
    rng = np.random.default_rng(SEED + k)
    shuffled = (
        pd.DataFrame({"tf": rng.choice(tfs, len(late)), "target": rng.choice(universe, len(late))})
        .groupby("tf")["target"]
        .apply(set)
    )
    overlaps.append(len(real & coregulated(shuffled)))

print(f"real {TIMES[-1]} (TF, complex) pairs : {len(real):,}")
print(f"mean overlap over {N_PERMUTATIONS} nulls  : {np.mean(overlaps):.2f}  (max {max(overlaps)})")

real 96h (TF, complex) pairs : 278
mean overlap over 50 nulls  : 0.02  (max 1)
